In [2]:
import numpy as np
import soundfile as sf

sr = 16000  # sample rate
duration = 6  # seconds
t = np.linspace(0, duration, int(sr*duration))

# Example melody: C4, D4, E4, G4, C5 (frequencies in Hz)
freqs = [261.63, 293.66, 329.63, 392.00, 523.25]
melody = np.zeros_like(t)

segment_len = len(t)//len(freqs)
for i, f in enumerate(freqs):
    start = i*segment_len
    end = (i+1)*segment_len
    melody[start:end] = 0.1 * np.sin(2*np.pi*f*t[start:end])  # low volume

sf.write(r"D:\shree\Miniproject\voicetomusic\Voice2Music\data\humming.wav", melody, sr)
print("Done ...")


Done ...


In [4]:
# voice2music_strict_conditioning_windows_fixed.py

import os
import numpy as np
import librosa
import crepe
import pretty_midi
from magenta.models.music_vae import configs
from magenta.models.music_vae.trained_model import TrainedModel
from magenta.music import midi_io
import soundfile as sf
import subprocess

# -------------------- USER CONFIG --------------------
voice_path = r"D:\shree\Miniproject\voicetomusic\Voice2Music\data\test.wav"
checkpoint_path = r"D:\shree\Miniproject\voicetomusic\Voice2Music\models\music_vae\hierdec-trio_16bar\hierdec-trio_16bar.ckpt"
soundfont_path = r"D:\shree\Miniproject\voicetomusic\Voice2Music\models\TimGM6mb.sf2"
fluidsynth_exe = r"D:\shree\Miniproject\voicetomusic\Voice2Music\models\fluidsynth\bin\fluidsynth.exe"

output_dir = r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae"
os.makedirs(output_dir, exist_ok=True)

input_midi_path = os.path.join(output_dir, "test.mid")
generated_trio_midi = os.path.join(output_dir, "generated_trio_uncondtest.mid")
combined_midi = os.path.join(output_dir, "combined_trio_test.mid")
generated_audio = os.path.join(output_dir, "generated_trio_test.wav")
final_song = os.path.join(output_dir, "final_song_test.wav")
# -----------------------------------------------------

import subprocess
import os

def render_midi_silently(midi_file, wav_file, sf2_file, fluidsynth_path):
    """
    Render MIDI -> WAV silently on Windows using FluidSynth.
    Prints detailed errors for debugging.
    """
    # Make sure paths exist
    if not os.path.exists(midi_file):
        raise FileNotFoundError(f"MIDI file not found: {midi_file}")
    if not os.path.exists(sf2_file):
        raise FileNotFoundError(f"SoundFont file not found: {sf2_file}")
    if not os.path.exists(fluidsynth_path):
        raise FileNotFoundError(f"FluidSynth executable not found: {fluidsynth_path}")

    # Build command (correct order: -F wav, sf2, midi)
    cmd = [
        fluidsynth_path,
        "-ni",          # no interactive mode
        "-a", "null",   # disable audio output
        "-F", wav_file, # output WAV file
        sf2_file,       # SoundFont
        midi_file,      # MIDI file
        "-r", "44100"   # sample rate
    ]

    # Print command for debugging
    print("Running FluidSynth command:", " ".join(f'"{c}"' if " " in c else c for c in cmd))

    # Run process
    result = subprocess.run(cmd, capture_output=True, text=True)

    # Print stdout & stderr
    if result.stdout.strip():
        print("FluidSynth stdout:\n", result.stdout)
    if result.stderr.strip():
        print("FluidSynth stderr:\n", result.stderr)

    # Check for errors
    if result.returncode != 0:
        raise RuntimeError(f"FluidSynth failed to render WAV. Return code: {result.returncode}")

    if not os.path.exists(wav_file):
        raise FileNotFoundError(f"Output WAV file not created: {wav_file}")

    print(f"Rendered MIDI to WAV successfully: {wav_file}")


# --- Step 1: load humming and extract pitch ---
y, sr = librosa.load(voice_path, sr=16000)
time, frequency, confidence, activation = crepe.predict(y, sr, viterbi=True)

# Filter low-confidence frames
conf_thresh = 0.2
frequency_filtered = np.where(confidence >= conf_thresh, frequency, np.nan)
midi_notes = np.round(69 + 12 * np.log2(frequency_filtered / 440.0))
midi_notes = midi_notes[np.isfinite(midi_notes)]

if len(midi_notes) == 0:
    raise RuntimeError("No melody notes found from humming. Try a clearer recording.")

# --- Step 2: create a quantized MIDI melody file ---
pm = pretty_midi.PrettyMIDI()
melody_inst = pretty_midi.Instrument(program=0, is_drum=False)
start = 0.0
note_dur = 0.5
for n in midi_notes:
    pitch = int(np.clip(n, 21, 108))
    note = pretty_midi.Note(velocity=100, pitch=pitch, start=start, end=start+note_dur)
    melody_inst.notes.append(note)
    start += note_dur
pm.instruments.append(melody_inst)
pm.write(input_midi_path)
print(f"Saved extracted melody MIDI to: {input_midi_path}")

# --- Step 3: load MusicVAE hierdec-trio and generate unconditional trio ---
config = configs.CONFIG_MAP['hierdec-trio_16bar']
model = TrainedModel(config, batch_size=1, checkpoint_dir_or_path=checkpoint_path)

print("Sampling MusicVAE (unconditional) to get bass+drums scaffolding...")
generated_seq = model.sample(n=1, length=256, temperature=0.8)[0]
midi_io.sequence_proto_to_midi_file(generated_seq, generated_trio_midi)
print(f"Saved generated trio (unconditional) to: {generated_trio_midi}")

# --- Step 4: combine - replace generated melody track with our melody ---
gen_pm = pretty_midi.PrettyMIDI(generated_trio_midi)
input_pm = pretty_midi.PrettyMIDI(input_midi_path)

melody_inst_index = None
for i, inst in enumerate(gen_pm.instruments):
    if not inst.is_drum:
        melody_inst_index = i
        break
if melody_inst_index is None:
    melody_inst_index = 0

print(f"Replacing generated instrument {melody_inst_index} notes with input melody.")
target_inst = gen_pm.instruments[melody_inst_index]
target_inst.notes = []

src_inst = input_pm.instruments[0]
for n in src_inst.notes:
    if n.start < gen_pm.get_end_time():
        end_time = min(n.end, gen_pm.get_end_time())
        target_inst.notes.append(pretty_midi.Note(velocity=100, pitch=n.pitch, start=n.start, end=end_time))

gen_pm.write(combined_midi)
print(f"Saved combined MIDI (melody + accompaniment) to: {combined_midi}")

# --- Step 5: render MIDI -> WAV using FluidSynth silently ---
render_midi_silently(combined_midi, generated_audio, soundfont_path, fluidsynth_exe)

# --- Step 6: Mix voice + accompaniment (voice clearly audible) ---
voice_audio, vsr = sf.read(voice_path)
music_audio, msr = sf.read(generated_audio)

# Resample if needed
if vsr != msr:
    voice_audio = librosa.resample(voice_audio.astype(np.float32), orig_sr=vsr, target_sr=msr)
    vsr = msr

# Mono-mix if stereo
if voice_audio.ndim > 1:
    voice_audio = np.mean(voice_audio, axis=1)
if music_audio.ndim > 1:
    music_audio = np.mean(music_audio, axis=1)

minlen = min(len(voice_audio), len(music_audio))
mixed = voice_audio[:minlen] + music_audio[:minlen] * 0.8  # voice clearly audible

# Normalize
mx = np.max(np.abs(mixed))
if mx > 1.0:
    mixed = mixed / mx

sf.write(final_song, mixed.astype(np.float32), msr)
print(f"Saved final mix to: {final_song}")

print("Done. Files produced:")
print(f" - Extracted melody MIDI: {input_midi_path}")
print(f" - Generated unconditional trio MIDI: {generated_trio_midi}")
print(f" - Combined final MIDI: {combined_midi}")
print(f" - Rendered accompaniment WAV: {generated_audio}")
print(f" - Final mixed WAV: {final_song}")


D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


47/47 [==============================] - 62s 1s/step
Saved extracted melody MIDI to: D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\test.mid
INFO:tensorflow:Building MusicVAE model with BidirectionalLstmEncoder, HierarchicalLstmDecoder, and hparams:
{'max_seq_len': 256, 'z_size': 512, 'free_bits': 256, 'max_beta': 0.2, 'beta_rate': 0.0, 'batch_size': 1, 'grad_clip': 1.0, 'clip_mode': 'global_norm', 'grad_norm_clip_to_zero': 10000, 'learning_rate': 0.001, 'decay_rate': 0.9999, 'min_learning_rate': 1e-05, 'conditional': True, 'dec_rnn_size': [1024, 1024], 'enc_rnn_size': [2048, 2048], 'dropout_keep_prob': 1.0, 'sampling_schedule': 'constant', 'sampling_rate': 0.0, 'use_cudnn': False, 'residual_encoder': False, 'residual_decoder': False, 'control_preprocessing_rnn_size': [256]}
INFO:tensorflow:
Encoder Cells (bidirectional):
  units: [2048, 2048]

INFO:tensorflow:
Hierarchical Decoder:
  input length: 256
  level output lengths: [16, 16]

INFO:tensorflow:
Decoder Cells:
  

D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\magenta\models\music_vae\lstm_utils.py:94: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  tf.layers.dense(
D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\magenta\contrib\rnn.py:750: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use the `layer.add_weight()` method instead.
  self._kernel = self.add_variable(
D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\magenta\contrib\rnn.py:752: UserWarning: `layer.add_variable` is deprecated and will be removed in a future version. Please use the `layer.add_weight()` method instead.
  self._bias = self.add_variable(


Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Please use `keras.layers.Bidirectional(keras.layers.RNN(cell))`, which is equivalent to this API
Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
`scale_identity_multiplier` is deprecated; please combine it into `scale_diag` directly instead.


D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\magenta\models\music_vae\base_model.py:195: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  mu = tf.layers.dense(
D:\shree\Miniproject\voicetomusic\Magenta_env\lib\site-packages\magenta\models\music_vae\base_model.py:200: UserWarning: `tf.layers.dense` is deprecated and will be removed in a future version. Please use `tf.keras.layers.Dense` instead.
  sigma = tf.layers.dense(


INFO:tensorflow:Restoring parameters from D:\shree\Miniproject\voicetomusic\Voice2Music\models\music_vae\hierdec-trio_16bar\hierdec-trio_16bar.ckpt
Sampling MusicVAE (unconditional) to get bass+drums scaffolding...
Saved generated trio (unconditional) to: D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\generated_trio_uncondtest.mid
Replacing generated instrument 0 notes with input melody.
Saved combined MIDI (melody + accompaniment) to: D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\combined_trio_test.mid
Running FluidSynth command: D:\shree\Miniproject\voicetomusic\Voice2Music\models\fluidsynth\bin\fluidsynth.exe -ni -a null -F D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\generated_trio_test.wav D:\shree\Miniproject\voicetomusic\Voice2Music\models\TimGM6mb.sf2 D:\shree\Miniproject\voicetomusic\Voice2Music\output\music_vae\combined_trio_test.mid -r 44100
FluidSynth stdout:
 fluidsynth: error: fluid_is_soundfont(): fopen() failed: 'File d